# BLEU, ROUGE, and Token F1

Traditional NLP gives us a set of well-understood metrics for comparing generated text against a reference: BLEU (translation, precision-focused), ROUGE (summarization, recall-focused), and Token F1 (set-overlap of words). They are cheap, fast, and well-studied — useful as cheap complements to semantic similarity or LLM judges.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import os
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## BLEU (Bilingual Evaluation Understudy)

BLEU was originally designed for machine translation. It compares n-gram overlap between the prediction and one or more references. An n-gram is just a sequence of `n` consecutive words: unigrams (`"The"`), bigrams (`"The cat"`), trigrams (`"The cat sat"`), and so on. BLEU measures **precision** — how many of the predicted n-grams appear in the reference — with a brevity penalty so the model can't game the score by producing very short outputs.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

def bleu_metric(example, pred, trace=None, threshold=0.5):
    reference = [example.answer.split()]  # List of word lists
    candidate = pred.answer.split()
    score = sentence_bleu(reference, candidate)
    return score >= threshold


example = dspy.Example(answer="The cat sat on the mat")
pred = dspy.Prediction(answer="The cat sat on the mat")

print(bleu_metric(example, pred))

## ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

ROUGE was developed for summarization. Where BLEU rewards the prediction for matching the reference, ROUGE rewards the prediction for *covering* the reference. ROUGE-L uses the longest common subsequence between prediction and reference, and the metric below uses its F1 form.

In [ ]:
from rouge import Rouge

def rouge_metric(example, pred, trace=None, threshold=0.5):
    rouge = Rouge()
    scores = rouge.get_scores(pred.answer, example.answer)[0]
    rouge_l = scores['rouge-l']['f']  # F1 score of longest common subsequence
    return rouge_l >= threshold


example = dspy.Example(
    answer="The quick brown fox jumps over the lazy dog."
)
pred = dspy.Prediction(
    answer="A quick brown fox jumped over the lazy dog."
)

print(rouge_metric(example, pred))

## Token F1

Token F1 treats the prediction and reference as sets of words and computes precision and recall on the overlap, then combines them into the harmonic mean (F1). It's ideal for extractive question answering where the answer is a short phrase pulled from a passage.

- **Precision**: of the words I predicted, how many were correct?
- **Recall**: of the words I should have predicted, how many did I find?
- **F1**: the balance — forces the prediction to be both accurate and complete.

In [ ]:
def token_f1(example, pred, trace=None):
    gold_tokens = set(example.answer.lower().split())
    pred_tokens = set(pred.answer.lower().split())

    if not pred_tokens or not gold_tokens:
        return 0.0

    precision = len(gold_tokens & pred_tokens) / len(pred_tokens)
    recall = len(gold_tokens & pred_tokens) / len(gold_tokens)

    if precision + recall == 0:
        return 0.0

    f1 = 2 * (precision * recall) / (precision + recall)
    return f1


example = dspy.Example(answer="barack hussein obama")
pred = dspy.Prediction(answer="barack obama")

print(token_f1(example, pred))

These traditional NLP metrics tend to underperform semantic similarity for most DSPy tasks, but they're nearly free to run and well-validated in the academic literature — worth keeping in your toolkit, especially when you need a fast sanity check during optimization.